# GPU Experiment 2: Long Generation (max_new_tokens=800)
This notebook proves that the metrics hold up when outputs naturally terminate (EOS Hit > 90%), addressing Reviewer's truncation concern.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes evaluate bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 16.8 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import numpy as np
from evaluate import load
from tqdm import tqdm
import time

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
)
bertscore = load("bertscore")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [3]:
# Load previously extracted steering vector
vec_data = torch.load('/kaggle/input/datasets/anhemgithom/steering-phase1-artifacts/v_steer.pt', map_location='cpu')
if isinstance(vec_data, dict):
    v_steer = vec_data.get('steering_vector', vec_data.get('v_steer', list(vec_data.values())[0]))
else:
    v_steer = vec_data
v_steer = v_steer.to(model.device).to(torch.float16)
print("Loaded vector successfully!")

# Load Test Data
with open('/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
test_data = data[-500:]

def format_prompt(q):
    messages = [{"role": "system", "content": "You are a helpful and accurate medical assistant."}, {"role": "user", "content": q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


Loaded vector successfully!


In [4]:
def evaluate_long(use_steering=False, alpha=18.0, K=16):
    hook_handle = None
    if use_steering:
        def steering_hook(module, input, output):
            t = output[0].shape[1] - input_length
            if 1 <= t <= K:
                output[0][:, -1, :] += alpha * (1 - (t-1)/K) * v_steer
            return output
        hook_handle = model.model.layers[8].register_forward_hook(steering_hook)
        
    generated, refs, hals, eos_hits = [], [], [], []
    
    for item in tqdm(test_data, desc="Evaluating"):
        prompt = format_prompt(item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        global input_length
        input_length = inputs.input_ids.shape[1]
        
        # KEY CHANGE: max_new_tokens=800
        out = model.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        
        gen_tokens = out[0][input_length:]
        eos_hits.append(tokenizer.eos_token_id in gen_tokens or tokenizer.pad_token_id in gen_tokens)
        
        generated.append(tokenizer.decode(gen_tokens, skip_special_tokens=True))
        refs.append(item.get('right_answer', item.get('positive_answer')))
        hals.append(item['hallucinated_answer'])
        
    if hook_handle: hook_handle.remove()
    
    # Compute BERTScore
    bs_ref = bertscore.compute(predictions=generated, references=refs, model_type="bert-base-multilingual-cased")['f1']
    bs_hal = bertscore.compute(predictions=generated, references=hals, model_type="bert-base-multilingual-cased")['f1']
    
    correct = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h)
    
    return {
        "correct": correct,
        "total": len(test_data),
        "accuracy": correct / len(test_data) * 100,
        "eos_hit_rate": sum(eos_hits) / len(eos_hits) * 100,
        "mean_bs": np.mean(bs_ref)
    }

print("Running Baseline (800 tokens)...")
base_res = evaluate_long(use_steering=False)

print("Running Steered (800 tokens)...")
steer_res = evaluate_long(use_steering=True)

print("\n=== LONG GENERATION RESULTS ===")
print(f"Baseline: Acc={base_res['accuracy']:.2f}%, EOS Hit={base_res['eos_hit_rate']:.2f}%, BS={base_res['mean_bs']:.4f}")
print(f"Steered : Acc={steer_res['accuracy']:.2f}%, EOS Hit={steer_res['eos_hit_rate']:.2f}%, BS={steer_res['mean_bs']:.4f}")


Running Baseline (800 tokens)...



Evaluating: 100%|██████████| 500/500 [2:52:51<00:00, 20.74s/it]


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running Steered (800 tokens)...



Evaluating: 100%|██████████| 500/500 [2:50:36<00:00, 20.47s/it]



=== LONG GENERATION RESULTS ===
Baseline: Acc=64.80%, EOS Hit=100.00%, BS=0.6804
Steered : Acc=64.80%, EOS Hit=100.00%, BS=0.6804
